# 20 Questions Game (Updated)
So here is an updated version of the game that features an entropy-based question selection to further reduce the number of questions.

**v1.5 update:** Added reasoning trace — each question now shows its entropy score and how many candidates remain after the answer. This makes the information-theoretic decision process visible.

First let's import the libraries and files that I will be using. Since we are working with csv's, I will be using pandas to load the files as a dataframe. I will also be using numpy for the entropy algo.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load data from csv files
df = pd.read_csv("data/word_attribute.csv")
qdf = pd.read_csv("data/attribute_question.csv")

Let's check to see if the data loaded properly

In [ ]:
df.head()

In [ ]:
qdf.head()

In [ ]:
# Convert questions from questions csv to dictionary
questions = dict(zip(qdf["Attributes"], qdf["Questions"]))

In [ ]:
# Prepare object and attribute list
objects = df["Name"].tolist()
attributes = df.columns[2:]

## Functions

In [ ]:
def binary_entropy(p):
    """Shannon binary entropy. Returns 0 at certainty (p=0 or p=1), max 1.0 at p=0.5."""
    if p in [0, 1]:
        return 0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)


def choose_best_question(df_subset, asked):
    """
    Selects the attribute with the highest entropy score from the current candidate set.
    High entropy means the question will split candidates most evenly — maximizing information gain.
    Returns the best attribute name and its entropy score.
    """
    best_attr = None
    max_entropy = -1

    for attr in attributes:
        if attr in asked:
            continue
        p = df_subset[attr].mean()
        entropy = binary_entropy(p)
        if entropy > max_entropy:
            max_entropy = entropy
            best_attr = attr

    return best_attr, max_entropy

## Game

The reasoning trace after each answer shows:
- **Entropy** — how informative the question was (1.0 = perfectly even split, 0.0 = no new info)
- **Candidates remaining** — how many objects are still possible after your answer

In [ ]:
# Start game loop
print("Think of one of these objects:")
print(", ".join(objects))
print("\nAnswer the following questions with 'y' or 'n'.")
print("-" * 50)

filtered_df = df.copy()
asked = set()
question_count = 0

while len(filtered_df) > 1 and len(asked) < len(attributes):
    attr, entropy_score = choose_best_question(filtered_df, asked)
    if attr is None:
        break

    question_count += 1
    question = questions.get(attr)
    answer = input(f"Q{question_count}: {question} (y/n): ").strip().lower()

    if answer not in ["y", "n"]:
        print("  Invalid input. Please answer with 'y' or 'n'.")
        continue

    expected = (answer == "y")
    filtered_df = filtered_df[filtered_df[attr] == expected]
    asked.add(attr)

    # Reasoning trace
    print(f"  → Entropy: {entropy_score:.3f} | Candidates remaining: {len(filtered_df)}",
          f"({', '.join(filtered_df['Name'].tolist())}" + ")" if len(filtered_df) <= 5 else "")

    if len(filtered_df) == 1:
        break

print("-" * 50)

# Final result
if len(filtered_df) == 1:
    guess = filtered_df["Name"].values[0]
    print(f"\nI guess you are thinking of: {guess}")
    print(f"Got it in {question_count} question(s).")
elif len(filtered_df) > 1:
    print("\nI'm not certain yet, but here are my best guesses:")
    print(", ".join(filtered_df["Name"].tolist()))
else:
    print("\nI couldn't guess your object. Maybe it's not in the database or the answers didn't match.")